In [1]:
from azure.storage.blob import BlobServiceClient
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
import io, os

In [3]:
# Connect to Azure Blob

os.environ["AZURE_CONNECTION_STRING"] = "ENTER_YOUR_CONNECTION_STRING_HERE"

connection_str = os.getenv("AZURE_CONNECTION_STRING")
client = BlobServiceClient.from_connection_string(connection_str) 
container = client.get_container_client("flight-data") 
 
def load_blob_csv(filename, sep=","):
    blob = container.get_blob_client(filename)
    data = blob.download_blob().readall()
    return pd.read_csv(io.BytesIO(data), sep=sep)

#load data set

df_eco  = load_blob_csv("economy.csv")
df_biz = load_blob_csv("business.csv", sep="\t")

print(df_eco.shape, df_biz.shape)

(206774, 11) (93487, 11)


In [4]:
df_eco.dtypes

date          object
airline       object
ch_code       object
num_code       int64
dep_time      object
from          object
time_taken    object
stop          object
arr_time      object
to            object
price         object
dtype: object

In [5]:
df_biz.dtypes

date          object
airline       object
ch_code       object
num_code       int64
dep_time      object
from          object
time_taken    object
stop          object
arr_time      object
to            object
price         object
dtype: object

In [6]:
# Add class column 
df_eco["class"] = "Economy" 
df_biz["class"] = "Business"

In [7]:
df_eco.head(1)

,date,airline,ch_code,num_code,dep_time,from,time_taken,stop,arr_time,to,price,class
0,11/02/2025,SpiceJet,SG,8709,18:55,Delhi,02h 10m,non-stop,21:05,Mumbai,"5,953",Economy


In [8]:
df_biz.head(1)

,date,airline,ch_code,num_code,dep_time,from,time_taken,stop,arr_time,to,price,class
0,11/02/2025,Air India,AI,868,18:00,Delhi,02h 00m,non-stop,20:00,Mumbai,"25,612",Business


In [9]:
# Combine into one dataframe 
df_raw = pd.concat([df_eco, df_biz], ignore_index=True) 
print(f"Total rows: {len(df_raw)}") 
print(df_raw.head(5))

Total rows: 300261
         date   airline ch_code  num_code dep_time   from time_taken  \
0  11/02/2025  SpiceJet      SG      8709    18:55  Delhi    02h 10m   
1  11/02/2025  SpiceJet      SG      8157     6:20  Delhi    02h 20m   
2  11/02/2025   AirAsia      I5       764     4:25  Delhi    02h 10m   
3  11/02/2025   Vistara      UK       995    10:20  Delhi    02h 15m   
4  11/02/2025   Vistara      UK       963     8:50  Delhi    02h 20m   

        stop arr_time      to  price    class  
0  non-stop     21:05  Mumbai  5,953  Economy  
1  non-stop      8:40  Mumbai  5,953  Economy  
2  non-stop      6:35  Mumbai  5,956  Economy  
3  non-stop     12:35  Mumbai  5,955  Economy  
4  non-stop     11:10  Mumbai  5,955  Economy  


In [11]:
missing = df_raw.isnull().sum() # Missing Values
print(missing)

date          0
airline       0
ch_code       0
num_code      0
dep_time      0
from          0
time_taken    0
stop          0
arr_time      0
to            0
price         0
class         0
dtype: int64


In [12]:
print(df_raw["price"].dtype) #checking data type of price

object


In [13]:
cat_cols = ['airline', 'from', 'stop', 'to']
for col in cat_cols:
    print(f'{col} ({df_raw[col].nunique()} unique): {sorted(df_raw[col].dropna().unique())}')
    print()

airline (8 unique): ['Air India', 'AirAsia', 'GO FIRST', 'Indigo', 'SpiceJet', 'StarAir', 'Trujet', 'Vistara']

from (6 unique): ['Bangalore', 'Chennai', 'Delhi', 'Hyderabad', 'Kolkata', 'Mumbai']

stop (40 unique): ['1-stop\n\t\t\t\t\t\t\t\t\t\t\t\t\n\t\t\t\t\t\t\t\t\t\t\t\t', '1-stop\n\t\t\t\t\t\t\t\t\t\t\t\tVia BBI\n\t\t\t\t\t\t\t\t\t\t\t\t', '1-stop\n\t\t\t\t\t\t\t\t\t\t\t\tVia Bhubaneswar\n\t\t\t\t\t\t\t\t\t\t\t\t', '1-stop\n\t\t\t\t\t\t\t\t\t\t\t\tVia Chennai\n\t\t\t\t\t\t\t\t\t\t\t\t', '1-stop\n\t\t\t\t\t\t\t\t\t\t\t\tVia Delhi\n\t\t\t\t\t\t\t\t\t\t\t\t', '1-stop\n\t\t\t\t\t\t\t\t\t\t\t\tVia GAU\n\t\t\t\t\t\t\t\t\t\t\t\t', '1-stop\n\t\t\t\t\t\t\t\t\t\t\t\tVia GAY\n\t\t\t\t\t\t\t\t\t\t\t\t', '1-stop\n\t\t\t\t\t\t\t\t\t\t\t\tVia GOP\n\t\t\t\t\t\t\t\t\t\t\t\t', '1-stop\n\t\t\t\t\t\t\t\t\t\t\t\tVia Guwahati\n\t\t\t\t\t\t\t\t\t\t\t\t', '1-stop\n\t\t\t\t\t\t\t\t\t\t\t\tVia HYD\n\t\t\t\t\t\t\t\t\t\t\t\t', '1-stop\n\t\t\t\t\t\t\t\t\t\t\t\tVia Hyderabad\n\t\t\t\t\t\t\t\t\t\t\t\t', '1-sto

In [14]:
#create a working copy
df = df_raw.copy()
print('Working copy created')

Working copy created


In [15]:
df["price"] = (
    df["price"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["price"] = pd.to_numeric(df["price"], errors="coerce")

In [17]:
print(df["price"].dtype)

int64


In [18]:
# Fix the stop column — clean up whitespace and newline characters 
df["stop"] = df["stop"].str.strip().str.split("\n").str[0].str.strip() 
# Map to simple categories 
df["stop"] = df["stop"].apply(lambda x: "non-stop" if "non" in x.lower() else "2+-stop" if "2" in x else "1-stop") 
print(df["stop"].value_counts())

1-stop      250929
non-stop     36044
2+-stop      13288
Name: stop, dtype: int64


In [19]:
# Parse duration to minutes
import re

def parse_duration(s):
    hours = re.search(r'(\d+)h', s)
    mins = re.search(r'(\d+)m', s)

    h = int(hours.group(1)) if hours else 0
    m = int(mins.group(1)) if mins else 0

    return h * 60 + m

df["duration_min"] = df["time_taken"].apply(parse_duration)

In [26]:
# Parse date
df["date"] = pd.to_datetime(df["date"], dayfirst=True)
print('Date dtype:', df['date'].dtype)
print('Date range:', df['date'].min().date(), '→', df['date'].max().date())

Date dtype: datetime64[ns]
Date range: 2025-02-11 → 2025-03-31


In [22]:
print('Data type (dep_time):', df['dep_time'].dtype)
print('Date dtype (arr_time):', df['arr_time'].dtype)

Data type (dep_time): object
Date dtype (arr_time): object


In [23]:
# Extract just the hour as a number from "HH:MM" format

df["dep_hour"] = df["dep_time"].str.split(":").str[0].astype(int)
df["arr_hour"] = df["arr_time"].str.split(":").str[0].astype(int)

In [24]:
df.head(1)

,date,airline,ch_code,num_code,dep_time,from,time_taken,stop,arr_time,to,price,class,duration_min,dep_hour,arr_hour
0,2025-02-11,SpiceJet,SG,8709,18:55,Delhi,02h 10m,non-stop,21:05,Mumbai,5953,Economy,130,18,21


In [25]:
# Convert dataframe to CSV in memory
csv_buffer = io.StringIO()
df.to_csv(csv_buffer, index=False)
csv_content = csv_buffer.getvalue()

# Upload to Azure Blob
blob_client = container.get_blob_client("processed/flights_clean.csv")
blob_client.upload_blob(csv_content, overwrite=True)

print("Saved to Azure Blob Storage!")
print("Location: flight-data/processed/flights_clean.csv")

Saved to Azure Blob Storage!
Location: flight-data/processed/flights_clean.csv
